# 🚢 TASK 10: Model Training & Save .pkl Files
## Complete Notebook for Google Colab
### Train XGBoost Model & Generate All .pkl Files

---
## ⚠️ IMPORTANT: Upload titanic.csv First!
1. Click Files button (left side)
2. Upload titanic.csv
3. Then run this notebook
---

## STEP 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import joblib
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully!')

## STEP 2: Upload & Load Data

In [ ]:
from google.colab import files

print('📥 Select titanic.csv to upload:')
uploaded = files.upload()

df = pd.read_csv('titanic.csv')

print('\n' + '='*80)
print('✅ DATA LOADED')
print('='*80)
print(f'Shape: {df.shape}')
print(f'Rows: {len(df)}')
print(f'Columns: {df.shape[1]}')
print(f'\nFirst 5 rows:')
print(df.head())
print(f'\nMissing values:')
print(df.isnull().sum())

## STEP 3: Feature Engineering

In [ ]:
print('\n' + '='*80)
print('🔧 FEATURE ENGINEERING')
print('='*80)

df_clean = df.copy()

# Feature 1: Family Size
print('\n1️⃣  Creating FamilySize...')
df_clean['FamilySize'] = df['SibSp'] + df['Parch'] + 1
print(f'   ✅ Range: {df_clean["FamilySize"].min()} to {df_clean["FamilySize"].max()}')

# Feature 2: Title from Name
print('\n2️⃣  Extracting Title from Name...')
df_clean['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
title_mapping = {'Mr': 'Mr', 'Mrs': 'Mrs', 'Miss': 'Miss', 'Master': 'Master'}
df_clean['Title'] = df_clean['Title'].map(title_mapping).fillna('Other')
print(f'   ✅ Titles: {df_clean["Title"].unique().tolist()}')

# Feature 3: IsAlone
print('\n3️⃣  Creating IsAlone feature...')
df_clean['IsAlone'] = (df_clean['FamilySize'] == 1).astype(int)
print(f'   ✅ Alone: {(df_clean["IsAlone"]==1).sum()}, With family: {(df_clean["IsAlone"]==0).sum()}')

print('\n✅ Feature engineering complete!')

## STEP 4: Data Cleaning

In [ ]:
print('\n' + '='*80)
print('🧹 DATA CLEANING')
print('='*80)

print('\nDropping unnecessary columns...')
df_clean = df_clean.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
print('✅ Dropped: PassengerId, Name, Ticket, Cabin')

print('\nHandling missing values...')
print(f'Before: {df_clean.isnull().sum().sum()} missing values')

df_clean['Age'].fillna(df_clean['Age'].median(), inplace=True)
print(f'   ✅ Age: Filled with median ({df_clean["Age"].median():.1f})')

df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0], inplace=True)
print(f'   ✅ Embarked: Filled with mode ({df_clean["Embarked"].mode()[0]})')

print(f'\nAfter: {df_clean.isnull().sum().sum()} missing values ✅')
print(f'\nDataframe shape: {df_clean.shape}')

## STEP 5: Encoding Categorical Variables

In [ ]:
print('\n' + '='*80)
print('🔢 ENCODING CATEGORICAL VARIABLES')
print('='*80)

# Create encoders
le_sex = LabelEncoder()
le_embarked = LabelEncoder()
le_title = LabelEncoder()

# Encode Sex
print('\n1️⃣  Encoding Sex...')
df_clean['Sex'] = le_sex.fit_transform(df_clean['Sex'])
print(f'   Mapping: {dict(zip(le_sex.classes_, le_sex.transform(le_sex.classes_)))}')

# Encode Embarked
print('\n2️⃣  Encoding Embarked...')
df_clean['Embarked'] = le_embarked.fit_transform(df_clean['Embarked'])
print(f'   Mapping: {dict(zip(le_embarked.classes_, le_embarked.transform(le_embarked.classes_)))}')

# Encode Title
print('\n3️⃣  Encoding Title...')
df_clean['Title'] = le_title.fit_transform(df_clean['Title'])
print(f'   Mapping: {dict(zip(le_title.classes_, le_title.transform(le_title.classes_)))}')

print('\n✅ All categorical variables encoded!')

## STEP 6: Prepare Features & Target

In [ ]:
print('\n' + '='*80)
print('📊 FEATURE PREPARATION')
print('='*80)

X = df_clean.drop('Survived', axis=1)
y = df_clean['Survived']

print(f'\nFeatures (X):   {X.shape}')
print(f'Target (y):     {y.shape}')
print(f'\nFeature columns ({X.shape[1]} total):')
for i, col in enumerate(X.columns, 1):
    print(f'   {i}. {col}')

print(f'\nTarget distribution:')
print(f'   Not Survived (0): {(y==0).sum()} ({(y==0).sum()/len(y)*100:.1f}%)')
print(f'   Survived (1):     {(y==1).sum()} ({(y==1).sum()/len(y)*100:.1f}%)')

## STEP 7: Feature Scaling

In [ ]:
print('\n' + '='*80)
print('⚖️  FEATURE SCALING')
print('='*80)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'\nScaling complete!')
print(f'   Mean: {X_scaled.mean():.6f}')
print(f'   Std Dev: {X_scaled.std():.6f}')
print(f'   Min: {X_scaled.min():.2f}')
print(f'   Max: {X_scaled.max():.2f}')
print(f'\n✅ All features normalized to standard scale!')

## STEP 8: Train-Test Split

In [ ]:
print('\n' + '='*80)
print('📊 TRAIN-TEST SPLIT')
print('='*80)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTraining set:   {len(X_train)} samples')
print(f'   Survived: {(y_train==1).sum()} ({(y_train==1).sum()/len(y_train)*100:.1f}%)')
print(f'   Not Survived: {(y_train==0).sum()} ({(y_train==0).sum()/len(y_train)*100:.1f}%)')

print(f'\nTest set:       {len(X_test)} samples')
print(f'   Survived: {(y_test==1).sum()} ({(y_test==1).sum()/len(y_test)*100:.1f}%)')
print(f'   Not Survived: {(y_test==0).sum()} ({(y_test==0).sum()/len(y_test)*100:.1f}%)')

print(f'\n✅ Stratified split maintains class balance!')

## STEP 9: Train XGBoost Model

In [ ]:
print('\n' + '='*80)
print('🚀 TRAINING XGBOOST MODEL')
print('='*80)

print('\nInitializing XGBoost...')
model = XGBClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(len(y_train) - y_train.sum()) / y_train.sum(),
    eval_metric='logloss',
    verbosity=0
)

print('\nTraining in progress...')
print('(This may take 30-60 seconds)\n')

model.fit(X_train, y_train)

print('✅ Model trained successfully!')
print(f'\nModel details:')
print(f'   Type: XGBoost Classifier')
print(f'   Trees: 100')
print(f'   Max Depth: 6')
print(f'   Learning Rate: 0.1')

## STEP 10: Model Evaluation

In [ ]:
print('\n' + '='*80)
print('📈 MODEL EVALUATION')
print('='*80)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print('\n🎖️  PERFORMANCE METRICS:')
print(f'\n   Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'   Precision: {precision:.4f} ({precision*100:.2f}%)')
print(f'   Recall:    {recall:.4f} ({recall*100:.2f}%)')
print(f'   F1-Score:  {f1:.4f}')
print(f'   ROC-AUC:   {auc:.4f}')

print('\n✅ Model Performance Excellent!')

## STEP 11: ⭐ SAVE ALL .pkl FILES ⭐

In [ ]:
print('\n' + '='*80)
print('💾 SAVING .pkl FILES')
print('='*80)

print('\nSaving files...')

# Save model
joblib.dump(model, 'titanic_model.pkl')
print('   ✅ titanic_model.pkl')

# Save scaler
joblib.dump(scaler, 'scaler.pkl')
print('   ✅ scaler.pkl')

# Save encoders
joblib.dump(le_sex, 'le_sex.pkl')
print('   ✅ le_sex.pkl')

joblib.dump(le_embarked, 'le_embarked.pkl')
print('   ✅ le_embarked.pkl')

joblib.dump(le_title, 'le_title.pkl')
print('   ✅ le_title.pkl')

print('\n' + '='*80)
print('✅ ALL FILES SAVED SUCCESSFULLY!')
print('='*80)

## STEP 12: Download Files

In [ ]:
print('\n📥 DOWNLOAD YOUR FILES:')
print('\n1. Click on "Files" icon (left sidebar)')
print('2. You will see:')
print('   • titanic_model.pkl')
print('   • scaler.pkl')
print('   • le_sex.pkl')
print('   • le_embarked.pkl')
print('   • le_title.pkl')
print('\n3. Right-click each file → Download')
print('\n4. Save to your computer')
print('\n' + '='*80)
print('🎉 YOU ARE DONE!')
print('='*80)
print('\nNext steps:')
print('1. Download all 5 .pkl files')
print('2. Create a folder: titanic-ml-app')
print('3. Add app.py to folder')
print('4. Add .pkl files to folder')
print('5. Add requirements.txt')
print('6. Push to GitHub')
print('7. Deploy to Streamlit Cloud')
print('8. Share live link!')

## OPTIONAL: Verify Files Are Saved

In [ ]:
import os

print('\n✅ VERIFICATION:')
print('\nFiles saved in current directory:')

files = ['titanic_model.pkl', 'scaler.pkl', 'le_sex.pkl', 'le_embarked.pkl', 'le_title.pkl']

for file in files:
    if os.path.exists(file):
        size = os.path.getsize(file)
        size_kb = size / 1024
        print(f'   ✅ {file:25s} ({size_kb:6.1f} KB)')
    else:
        print(f'   ❌ {file:25s} (NOT FOUND)')

print('\n🎉 All files ready for download!')